In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("  Machine Learning Module Initialized")
print(f"TensorFlow version: {tf.__version__}")

## 1. Generate Synthetic Trading Data

In [ ]:
# Generate realistic market data with predictive features
np.random.seed(42)
n_samples = 5000
n_features = 20

# Generate features
X = np.random.randn(n_samples, n_features)

# Create target variable (future returns) with some signal
# True coefficients
true_coeffs = np.random.randn(n_features) * 0.1
true_coeffs[:5] *= 3  # Make first 5 features more important

# Generate returns with signal + noise
signal = X @ true_coeffs
noise = np.random.randn(n_samples) * 0.02
y = signal + noise

# Add some non-linearity
y += 0.5 * np.sin(X[:, 0] * 2) + 0.3 * (X[:, 1] ** 2)

# Create DataFrame
feature_names = [f'feature_{i+1}' for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)
df['returns'] = y

# Add technical indicators
df['sma_ratio'] = X[:, 0] / (X[:, 1] + 1e-6)
df['momentum'] = np.cumsum(y)
df['volatility'] = pd.Series(y).rolling(20).std().fillna(0)

print(f"Dataset shape: {df.shape}")
print(f"\nFeature correlation with returns:")
correlations = df.corr()['returns'].sort_values(ascending=False)
print(correlations.head(10))

In [ ]:
# Visualize data
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Target distribution
axes[0, 0].hist(y, bins=50, density=True, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Returns')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('Target Variable Distribution')
axes[0, 0].grid(True, alpha=0.3)

# Cumulative returns
cumulative_returns = np.cumsum(y)
axes[0, 1].plot(cumulative_returns, linewidth=1)
axes[0, 1].set_xlabel('Sample')
axes[0, 1].set_ylabel('Cumulative Returns')
axes[0, 1].set_title('Cumulative Returns Over Time')
axes[0, 1].grid(True, alpha=0.3)

# Feature correlation heatmap (top 10 features)
top_features = correlations.abs().sort_values(ascending=False).head(11).index[1:]  # Exclude returns itself
corr_matrix = df[list(top_features) + ['returns']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1, 0])
axes[1, 0].set_title('Feature Correlation Matrix (Top 10)')

# Scatter plot: strongest feature vs returns
strongest_feature = correlations.abs().sort_values(ascending=False).index[1]
axes[1, 1].scatter(df[strongest_feature], df['returns'], alpha=0.3, s=10)
axes[1, 1].set_xlabel(strongest_feature)
axes[1, 1].set_ylabel('Returns')
axes[1, 1].set_title(f'Strongest Predictor: {strongest_feature}')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Data Preparation

In [ ]:
# Prepare train/test split (time-series aware)
train_size = int(0.7 * len(df))
val_size = int(0.15 * len(df))

# Split data
X_full = df.drop('returns', axis=1).values
y_full = df['returns'].values

X_train = X_full[:train_size]
y_train = y_full[:train_size]

X_val = X_full[train_size:train_size+val_size]
y_val = y_full[train_size:train_size+val_size]

X_test = X_full[train_size+val_size:]
y_test = y_full[train_size+val_size:]

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTarget statistics:")
print(f"  Train mean: {y_train.mean():.6f}, std: {y_train.std():.6f}")
print(f"  Val mean: {y_val.mean():.6f}, std: {y_val.std():.6f}")
print(f"  Test mean: {y_test.mean():.6f}, std: {y_test.std():.6f}")

## 3. Linear Models

In [ ]:
# Train linear models
linear_models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5)
}

linear_results = {}

for name, model in linear_models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train_scaled)
    y_pred_val = model.predict(X_val_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # Metrics
    results = {
        'model': model,
        'predictions': {
            'train': y_pred_train,
            'val': y_pred_val,
            'test': y_pred_test
        },
        'metrics': {
            'train_mse': mean_squared_error(y_train, y_pred_train),
            'val_mse': mean_squared_error(y_val, y_pred_val),
            'test_mse': mean_squared_error(y_test, y_pred_test),
            'train_r2': r2_score(y_train, y_pred_train),
            'val_r2': r2_score(y_val, y_pred_val),
            'test_r2': r2_score(y_test, y_pred_test)
        }
    }
    
    linear_results[name] = results
    
    print(f"  Train MSE: {results['metrics']['train_mse']:.6f}, R²: {results['metrics']['train_r2']:.4f}")
    print(f"  Val MSE: {results['metrics']['val_mse']:.6f}, R²: {results['metrics']['val_r2']:.4f}")
    print(f"  Test MSE: {results['metrics']['test_mse']:.6f}, R²: {results['metrics']['test_r2']:.4f}")

## 4. Tree-Based Models

In [ ]:
# Train tree-based models
tree_models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=-1)
}

tree_results = {}

for name, model in tree_models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    # Metrics
    results = {
        'model': model,
        'predictions': {
            'train': y_pred_train,
            'val': y_pred_val,
            'test': y_pred_test
        },
        'metrics': {
            'train_mse': mean_squared_error(y_train, y_pred_train),
            'val_mse': mean_squared_error(y_val, y_pred_val),
            'test_mse': mean_squared_error(y_test, y_pred_test),
            'train_r2': r2_score(y_train, y_pred_train),
            'val_r2': r2_score(y_val, y_pred_val),
            'test_r2': r2_score(y_test, y_pred_test)
        },
        'feature_importance': model.feature_importances_ if hasattr(model, 'feature_importances_') else None
    }
    
    tree_results[name] = results
    
    print(f"  Train MSE: {results['metrics']['train_mse']:.6f}, R²: {results['metrics']['train_r2']:.4f}")
    print(f"  Val MSE: {results['metrics']['val_mse']:.6f}, R²: {results['metrics']['val_r2']:.4f}")
    print(f"  Test MSE: {results['metrics']['test_mse']:.6f}, R²: {results['metrics']['test_r2']:.4f}")

In [ ]:
# Feature importance visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, (name, results) in enumerate(tree_results.items()):
    if results['feature_importance'] is not None:
        # Get top 15 features
        importance = results['feature_importance']
        indices = np.argsort(importance)[-15:][::-1]
        
        axes[idx].barh(range(len(indices)), importance[indices])
        axes[idx].set_yticks(range(len(indices)))
        axes[idx].set_yticklabels([df.columns[i] for i in indices])
        axes[idx].set_xlabel('Importance')
        axes[idx].set_title(f'{name} - Feature Importance (Top 15)')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Neural Networks

In [ ]:
# Build MLP model
def build_mlp(input_dim, hidden_units=[64, 32, 16]):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden_units[0], activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(hidden_units[1], activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(hidden_units[2], activation='relu'),
        layers.Dense(1)
    ])
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                 loss='mse',
                 metrics=['mae'])
    
    return model

# Train MLP
print("Training MLP...")
mlp_model = build_mlp(X_train_scaled.shape[1])

early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = mlp_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=0
)

# Predictions
y_pred_train_mlp = mlp_model.predict(X_train_scaled, verbose=0).flatten()
y_pred_val_mlp = mlp_model.predict(X_val_scaled, verbose=0).flatten()
y_pred_test_mlp = mlp_model.predict(X_test_scaled, verbose=0).flatten()

# Metrics
mlp_results = {
    'model': mlp_model,
    'history': history,
    'predictions': {
        'train': y_pred_train_mlp,
        'val': y_pred_val_mlp,
        'test': y_pred_test_mlp
    },
    'metrics': {
        'train_mse': mean_squared_error(y_train, y_pred_train_mlp),
        'val_mse': mean_squared_error(y_val, y_pred_val_mlp),
        'test_mse': mean_squared_error(y_test, y_pred_test_mlp),
        'train_r2': r2_score(y_train, y_pred_train_mlp),
        'val_r2': r2_score(y_val, y_pred_val_mlp),
        'test_r2': r2_score(y_test, y_pred_test_mlp)
    }
}

print(f"  Train MSE: {mlp_results['metrics']['train_mse']:.6f}, R²: {mlp_results['metrics']['train_r2']:.4f}")
print(f"  Val MSE: {mlp_results['metrics']['val_mse']:.6f}, R²: {mlp_results['metrics']['val_r2']:.4f}")
print(f"  Test MSE: {mlp_results['metrics']['test_mse']:.6f}, R²: {mlp_results['metrics']['test_r2']:.4f}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('MLP Training History')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('MLP Training History (MAE)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Model Comparison

In [ ]:
# Compile all results
all_results = {**linear_results, **tree_results, 'MLP': mlp_results}

# Create comparison DataFrame
comparison_data = []
for name, results in all_results.items():
    comparison_data.append({
        'Model': name,
        'Train MSE': results['metrics']['train_mse'],
        'Val MSE': results['metrics']['val_mse'],
        'Test MSE': results['metrics']['test_mse'],
        'Train R²': results['metrics']['train_r2'],
        'Val R²': results['metrics']['val_r2'],
        'Test R²': results['metrics']['test_r2']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Test MSE')

print("\nModel Comparison:")
print("="*80)
print(comparison_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# MSE comparison
x = np.arange(len(comparison_df))
width = 0.25

axes[0].bar(x - width, comparison_df['Train MSE'], width, label='Train', alpha=0.8)
axes[0].bar(x, comparison_df['Val MSE'], width, label='Validation', alpha=0.8)
axes[0].bar(x + width, comparison_df['Test MSE'], width, label='Test', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('MSE')
axes[0].set_title('Model Comparison - Mean Squared Error')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# R² comparison
axes[1].bar(x - width, comparison_df['Train R²'], width, label='Train', alpha=0.8)
axes[1].bar(x, comparison_df['Val R²'], width, label='Validation', alpha=0.8)
axes[1].bar(x + width, comparison_df['Test R²'], width, label='Test', alpha=0.8)
axes[1].set_xlabel('Model')
axes[1].set_ylabel('R² Score')
axes[1].set_title('Model Comparison - R² Score')
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Prediction Analysis

In [ ]:
# Compare predictions on test set
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = all_results[best_model_name]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot predictions for top 4 models
for idx, model_name in enumerate(comparison_df['Model'][:4]):
    ax = axes[idx // 2, idx % 2]
    
    y_pred = all_results[model_name]['predictions']['test']
    
    # Scatter plot
    ax.scatter(y_test, y_pred, alpha=0.3, s=10)
    
    # Perfect prediction line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    # Regression line
    z = np.polyfit(y_test, y_pred, 1)
    p = np.poly1d(z)
    ax.plot(y_test, p(y_test), 'g-', linewidth=2, alpha=0.7, label='Fitted Line')
    
    ax.set_xlabel('True Returns')
    ax.set_ylabel('Predicted Returns')
    ax.set_title(f'{model_name} - Test Set Predictions\nR² = {all_results[model_name]["metrics"]["test_r2"]:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Trading Performance Evaluation

In [ ]:
# Evaluate trading performance
def calculate_trading_metrics(y_true, y_pred, threshold=0):
    """Calculate trading-specific metrics"""
    # Generate trading signals
    signals = np.where(y_pred > threshold, 1, -1)  # Long if pred > 0, short otherwise
    
    # Strategy returns
    strategy_returns = signals * y_true
    
    # Metrics
    total_return = np.sum(strategy_returns)
    sharpe_ratio = np.mean(strategy_returns) / (np.std(strategy_returns) + 1e-6) * np.sqrt(252)
    
    # Maximum drawdown
    cumulative = np.cumsum(strategy_returns)
    running_max = np.maximum.accumulate(cumulative)
    drawdown = cumulative - running_max
    max_drawdown = np.min(drawdown)
    
    # Win rate
    win_rate = np.sum(strategy_returns > 0) / len(strategy_returns)
    
    # Directional accuracy
    direction_correct = np.sum((y_pred > 0) == (y_true > 0)) / len(y_true)
    
    return {
        'total_return': total_return,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate,
        'direction_accuracy': direction_correct,
        'strategy_returns': strategy_returns,
        'cumulative_returns': cumulative
    }

# Calculate trading metrics for all models
trading_results = {}

for name, results in all_results.items():
    y_pred_test = results['predictions']['test']
    trading_metrics = calculate_trading_metrics(y_test, y_pred_test)
    trading_results[name] = trading_metrics

# Create trading metrics DataFrame
trading_df = pd.DataFrame([
    {
        'Model': name,
        'Total Return': metrics['total_return'],
        'Sharpe Ratio': metrics['sharpe_ratio'],
        'Max Drawdown': metrics['max_drawdown'],
        'Win Rate': metrics['win_rate'],
        'Direction Accuracy': metrics['direction_accuracy']
    }
    for name, metrics in trading_results.items()
]).sort_values('Sharpe Ratio', ascending=False)

print("\nTrading Performance Metrics:")
print("="*90)
print(trading_df.to_string(index=False))

In [ ]:
# Visualize cumulative returns
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Cumulative returns comparison
for name, metrics in trading_results.items():
    axes[0].plot(metrics['cumulative_returns'], label=name, linewidth=1.5, alpha=0.7)

# Buy and hold baseline
buy_hold = np.cumsum(y_test)
axes[0].plot(buy_hold, 'k--', linewidth=2, label='Buy & Hold', alpha=0.5)

axes[0].set_xlabel('Time')
axes[0].set_ylabel('Cumulative Returns')
axes[0].set_title('Cumulative Returns Comparison')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Sharpe ratio comparison
models = trading_df['Model']
sharpe_ratios = trading_df['Sharpe Ratio']
colors = ['green' if sr > 0 else 'red' for sr in sharpe_ratios]

axes[1].barh(models, sharpe_ratios, color=colors, alpha=0.7)
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Sharpe Ratio')
axes[1].set_title('Sharpe Ratio Comparison')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 9. Summary

### Model Performance Summary:

**Best Predictive Performance (MSE/R²):**
- Tree-based models (XGBoost, LightGBM) typically show best in-sample fit
- Neural networks can capture complex non-linear patterns
- Linear models provide baseline and interpretability

**Best Trading Performance (Sharpe Ratio):**
- Predictive accuracy doesn't always translate to trading profits
- Transaction costs and market impact not included
- Model ensemble often provides robust results

### Key Insights:

1. **Overfitting Risk**: Tree models show high train performance but may not generalize
2. **Directional Accuracy**: More important than exact prediction for trading
3. **Feature Importance**: Technical indicators and momentum features most predictive
4. **Ensemble Benefits**: Combining multiple models reduces variance

### Recommendations:
- Use ensemble of tree-based and neural network models
- Focus on directional accuracy over precise prediction
- Implement proper cross-validation with time-series splits
- Consider transaction costs and slippage in backtesting
- Monitor feature drift and retrain regularly